# SemIf · Inference

<a href="https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Inference.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Score runtime-defined decisions by reading the model's option logits directly: one forward pass per decision, no generated tokens, no answer parsing. This notebook uses Unsloth's fast model loading and left-padded batching instead of a serving engine.

To run this, press *Runtime* → *Run all* on a **free** Tesla T4 Colab instance.
Companion notebooks: [inference](https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Inference.ipynb) · [evaluation](https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Eval.ipynb) · [training](https://colab.research.google.com/github/furbyhaxx/semlf/blob/master/notebooks/SemIf_Train.ipynb).

### Installation

Same stack as Unsloth's Qwen3.5 (4B) notebook: Unsloth, its bundled gated-delta-net
kernels and `causal_conv1d` for the Qwen3.5 linear-attention layers. No vLLM:
every decision is one forward pass, so there is nothing to serve.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# Unsloth bundles the gated delta net kernels; a leftover pip fla would shadow them
!uv pip uninstall -qqq flash-linear-attention fla-core
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# @title Fetch SemIf { display-mode: "form" }
# SemIf is put on sys.path instead of pip-installed: its pyproject pins torch 2.10,
# which would replace the torch Unsloth was installed against.
import json, subprocess, sys, time
from pathlib import Path

SEMLF_REPO = "https://github.com/furbyhaxx/semlf"  # @param {type:"string"}
SEMLF_REF = "master"  # @param {type:"string"}

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "semif_phase1").is_dir()), None)
if ROOT is None:
    ROOT = (Path("/content") if Path("/content").is_dir() else Path.cwd()) / "semlf"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", SEMLF_REF, SEMLF_REPO, str(ROOT)], check=True)
for path in (ROOT / "src", ROOT / "notebooks"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
COMMIT = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"SemIf at {ROOT} ({COMMIT[:12] or 'not a git checkout'}); outputs go to {OUT}")

### Configuration

Pick a pinned model from the list or type any Hub ID, local checkpoint, or a LoRA adapter directory saved by the training notebook.

In [ ]:
# @title Model { display-mode: "form" }
MODELS = {  # pinned revisions; any other Hub model or local path also works
    "Qwen/Qwen3.5-4B": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",  # reference: committed BF16 predictions
    "Qwen/Qwen3.5-2B": "15852e8c16360a2fea060d615a32b45270f8a8fc",
    "Qwen/Qwen3.5-0.8B": "2fc06364715b967f1860aea9cf38778875588b17",
    "Qwen/Qwen3.5-9B": "c202236235762e1c871ad0ccb60c8ee5ba337b9a",
    "Qwen/Qwen3-4B-Instruct-2507": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "Qwen/Qwen3-1.7B": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
    "Qwen/Qwen3-0.6B": "c1899de289a04d12100db370d81485cdf75e47ca",
    "openbmb/MiniCPM5-2B": "12a3808a956f869c767195e9266b59c4d21d92e2",
}

MODEL = "Qwen/Qwen3.5-4B"  # @param ["Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-0.8B", "Qwen/Qwen3.5-9B", "Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B", "openbmb/MiniCPM5-2B"] {allow-input: true}
REVISION = ""  # @param {type:"string"}
WEIGHTS = "auto"  # @param ["auto", "16bit", "bnb-8bit", "bnb-4bit"]
MAX_SEQ_LENGTH = 4096  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
INPUT_JSONL = ""  # @param {type:"string"}

### GPU and precision

`WEIGHTS = "auto"` keeps 16-bit weights whenever they fit, because that preserves
the reference accuracy, and drops to 4-bit NF4 only when they do not. Choose it
yourself to trade accuracy for memory:

| Card | VRAM | Qwen3.5-4B suggestion |
| --- | --- | --- |
| A100 / L4 / RTX 3090-4090 | 22-80 GB | `16bit` in bf16, the reference setting |
| T4 / V100 / RTX 2080 Ti | 11-16 GB | `16bit` in fp16 (no bf16 before Ampere) |
| RTX 2060-3070, 8 GB | 6-8 GB | `bnb-4bit`, or a smaller base model |

`bnb-8bit` is the most faithful quantized format but is usually *slower* than
16-bit; `bnb-4bit` is smaller and faster than 8-bit. Unsloth's Triton kernels
need compute capability 7.0 or newer (V100, T4, RTX 20xx and later).

In [ ]:
import unsloth  # import before transformers so Unsloth can patch it
import torch
from huggingface_hub import model_info
from semif_phase1 import unsloth_backend as ub

GPU = ub.gpu_profile()
print(f"{GPU['name']}: {GPU['total_gib']} GiB, compute {GPU['compute_capability']}, bf16={GPU['bf16']}")
if float(GPU["compute_capability"]) < 7.0:
    print("WARNING: below compute 7.0; Unsloth's Triton kernels are unsupported on this card.")

revision = REVISION or MODELS.get(MODEL)
WEIGHTS_USED = WEIGHTS
if WEIGHTS == "auto":
    if Path(MODEL).exists():
        WEIGHTS_USED = "16bit"
    else:
        info = model_info(MODEL, revision=revision)
        params = info.safetensors.total if info.safetensors else None
        WEIGHTS_USED = ub.suggest_weights(params, GPU["total_bytes"]) if params else "16bit"
        print(f"{MODEL}: {params / 1e9:.2f}B parameters" if params else f"{MODEL}: parameter count unknown")
print(f"weights: {WEIGHTS_USED}")

### Load through Unsloth

Qwen3.5 checkpoints are vision-language models and load with `FastVisionModel`; text-only models use `FastLanguageModel`. The resolved commit, precision and GPU are recorded in every result row.

In [ ]:
model, tokenizer, META = ub.load_model(
    MODEL,
    revision,
    load_in_4bit = WEIGHTS_USED == "bnb-4bit",
    load_in_8bit = WEIGHTS_USED == "bnb-8bit",
    max_seq_length = MAX_SEQ_LENGTH,
)
META["semlf_commit"] = COMMIT
print(json.dumps(META, indent = 1))
print(f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB allocated after loading")

### Ask one decision

`decide` builds the same row format as `examples/decisions.jsonl` and returns option probabilities.

In [ ]:
from semif_phase1.unsloth_backend import score_rows, throughput

def decide(state, question, options, image = None):
    row = {"id": "adhoc", "state": state, "question": question,
           "options": [{"id": key, "description": text} for key, text in options.items()]}
    if image:
        row["Image"] = str(image)
    result = score_rows(model, tokenizer, [row], META, mode = "direct")[0]
    return {key: round(p, 4) for key, p in zip(result["option_ids"], result["probabilities"])}

decide(
    "Customer cannot access an account after a password reset.",
    "Which queue should handle this request?",
    {"access": "Account access support.", "billing": "Billing support."},
)

### Score a JSONL file in batches

Rows are sorted by length and left-padded so the answer position is always the last column. If a batch runs out of memory it is halved and retried, so a large `BATCH_SIZE` is safe on small cards.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from colab_helpers import read_jsonl, write_create_only

source = Path(INPUT_JSONL) if INPUT_JSONL else ROOT / "examples" / "decisions.jsonl"
rows = read_jsonl(source)
bar = tqdm(total = len(rows), desc = source.name)
start = time.perf_counter()
results = score_rows(model, tokenizer, rows, META, mode = "batched", batch_size = BATCH_SIZE, progress = bar.update)
bar.close()
print(throughput(results, time.perf_counter() - start))

pd.DataFrame([
    {"id": r["id"], "choice": r["option_ids"][max(range(len(r["probabilities"])), key = r["probabilities"].__getitem__)],
     "p": round(max(r["probabilities"]), 4), "slot_mass": round(r.get("allowed_token_mass", float("nan")), 4),
     "tokens": r["input_tokens"]}
    for r in results
])

In [ ]:
path = write_create_only(OUT / f"scores-{source.stem}-{time.strftime('%Y%m%d-%H%M%S')}.jsonl", results)
print(f"wrote {path}")

`slot_mass` is the share of the full-vocabulary probability that lands on the answer letters. Values far below 1 mean the model wanted to say something else, which is a warning sign when trying smaller base models.

### Many criteria over one state

The first 21 rows of the owned `shape777` fixture share one 8 KB state. Compare fresh per-row scoring, batching, and `shared` mode, which prefills the state once and branches every criterion from its KV cache.

In [ ]:
from semif_phase1.unsloth_backend import compare_predictions

group = []
with open(ROOT / "benchmarks" / "data" / "shape777.jsonl") as stream:
    for line in stream:
        row = json.loads(line)
        if group and row["state"] != group[0]["state"]:
            break
        group.append(row)

timings, by_mode = [], {}
for mode in ("direct", "batched", "shared"):
    try:
        score_rows(model, tokenizer, group[:2], META, mode = mode, batch_size = BATCH_SIZE)  # warm-up
        torch.cuda.synchronize()
        start = time.perf_counter()
        by_mode[mode] = score_rows(model, tokenizer, group, META, mode = mode, batch_size = BATCH_SIZE)
        torch.cuda.synchronize()
        wall = time.perf_counter() - start
        agreement = compare_predictions(by_mode["direct"], by_mode[mode])["argmax_agreement"]
        timings.append({"mode": mode, "seconds": round(wall, 3), "decisions/s": round(len(group) / wall, 2),
                        "argmax agreement with direct": agreement})
    except Exception as error:  # shared mode needs a native prefix cache that some patched models lack
        timings.append({"mode": mode, "error": f"{type(error).__name__}: {error}"[:200]})
pd.DataFrame(timings)

### Image decisions

Qwen3.5 checkpoints read an optional local `Image` path through their native processor. Image rows are scored one at a time.

In [ ]:
if META["loader"] == "FastVisionModel":
    image_rows = [dict(row, Image = str(ROOT / row["Image"])) for row in read_jsonl(ROOT / "examples" / "decisions-image.jsonl")]
    for result in score_rows(model, tokenizer, image_rows, META, mode = "direct"):
        print(result["id"], dict(zip(result["option_ids"], (round(p, 4) for p in result["probabilities"]))))
else:
    print(f"{MODEL} is text-only; skipping image rows")

### Notes

* Probabilities are conditional option scores over the supplied options, not
  calibrated confidences. Validate them on your own workload before acting on them.
* Outputs are create-only, matching the repository's benchmark policy: every run
  writes a new file under `outputs/`.
* The Unsloth install cell follows
  [unslothai/notebooks](https://github.com/unslothai/notebooks) (LGPL-3.0).